<a id="inhalt"></a>

# Ausführliche Zusammenfassung: King County House Prices

> ℹ️ **Über diese Zusammenfassung**
>
> - Sie erklärt alle Ideen des Projekts so, dass du sie verstehst, ohne die Notebooks durchzuarbeiten.
> - Sie ist nach Ideen geordnet, nicht nach Notebooks. Jeder Abschnitt nennt das passende Kapitel aus der [kurzen Zusammenfassung](zusammenfassung.ipynb).
> - Alle Zahlen stammen aus dem [Lösungs-Weg](loesungsweg.ipynb) und wurden neu berechnet.
> - Die Begriffe findest du im [Glossar](zusammenfassung.ipynb#glossar).
> - Schrift-Art und Schrift-Größe stellst du im Editor ein.

## Inhalt

| Nr. | Abschnitt | Kapitel |
| :--- | :--- | :--- |
| 1 | [Die Aufgabe: Regression und Metriken](#s1) | 2.1 |
| 2 | [Zwei Tabellen sicher verknüpfen](#s2) | 1 |
| 3 | [Trainings-Daten und Test-Daten](#s3) | 2.1 |
| 4 | [Die Daten untersuchen](#s4) | 2.1 |
| 5 | [Änderungen von Hand und neue Merkmale](#s5) | 2.1 |
| 6 | [Basis-Modelle als Messlatte](#s6) | 2.1 |
| 7 | [Pipeline und ColumnTransformer](#s7) | 2.2 |
| 8 | [Ridge, Regularisierung und Polynom-Merkmale](#s8) | 2.2 |
| 9 | [Kreuz-Validierung, Zufalls-Suche und Gitter-Suche](#s9) | 2.2 |
| 10 | [Bewerten auf den Test-Daten und Residuen](#s10) | 2.2 |
| 11 | [Typische Denkfehler](#denkfehler) | alle |
| 12 | [Code-Spickzettel nach Thema](#spick) | alle |
| 13 | [Prüfe dich selbst](#pruefen) | alle |
| 14 | [Der rote Faden in einem Satz](#faden) | alle |

## Die Datensätze

| Datensatz | Inhalt | Ziel-Variable | Einheiten | Notebook |
| :--- | :--- | :--- | :--- | :--- |
| `../data/king_county_house_sales.csv` | 21.597 Verkäufe vom 2014-05-02 bis 2015-05-27, mit `date`, `price`, `house_id` und der Verkaufs-Kennung `id` | `price` | Dollar | 01 |
| `../data/king_county_house_details.csv` | 21.420 Häuser mit 19 Spalten, zum Beispiel Flächen, Zimmer, Note, Lage | keine | Flächen in Quadrat-Fuß, Jahre, Grad für `lat` und `long` | 01 |
| `../data/eda.csv` | die verknüpfte Tabelle: 21.597 Zeilen, 23 Spalten, eine Zeile pro Verkauf | `price` | wie oben | 01 schreibt sie, 02 liest sie |

Das Projekt hat keine freiwilligen Zusatz-Notebooks. Die Aufgaben aus beiden Notebooks sind in den Abschnitten unten vollständig erklärt: Aufgaben 1 bis 6 aus Notebook 01 in Abschnitt 2, Aufgaben 1 bis 8 aus Notebook 02 in den Abschnitten 1 und 3 bis 10.

<a id="s1"></a>

## 1. Die Aufgabe: Regression und Metriken ([Kapitel 2.1](zusammenfassung.ipynb#k2-1))

**Die Idee.** Du sollst für jedes Haus eine Zahl vorhersagen, den Verkaufs-Preis. Eine solche Aufgabe heißt **Regression**. Die Spalte `price` ist die **Ziel-Variable**, alle anderen nutzbaren Spalten sind **Merkmale**. Bevor du ein Modell baust, legst du fest, woran du es misst. Diese Zahl ist die **Metrik**.

**Warum das funktioniert.** Ohne feste **Metrik** weißt du nicht, ob eine Änderung das Modell besser macht. Im Projekt stehen drei **Metriken** zur Auswahl:

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

In Worten: Du quadrierst jeden Fehler, nimmst den Mittelwert und ziehst dann die Wurzel, damit das Ergebnis wieder in Dollar ist.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

In Worten: Du nimmst den Mittelwert der Fehler ohne Vorzeichen.

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

In Worten: Das **R²** vergleicht deine quadrierten Fehler mit denen einer Vorhersage, die immer den mittleren Preis nimmt.

Der **RMSE** quadriert die Fehler, bevor er mittelt. Ein Fehler von 200.000 Dollar zählt deshalb viermal so stark wie einer von 100.000 Dollar. Das passt zu Haus-Preisen, denn große Fehler sind teuer. Der **MAE** ist leichter zu lesen, weil er einfach der typische Fehler in Dollar ist. Das **R²** hat keine Einheit und sagt, welcher Anteil der Preis-Unterschiede erklärt wird.

**Mit den echten Zahlen.** Das beste Modell hat auf den **Test-Daten** einen **RMSE** von 123.490 Dollar, einen **MAE** von 75.765 Dollar und ein **R²** von 0,88. Der **RMSE** ist größer als der **MAE**, weil einige sehr große Fehler im Quadrat besonders stark zählen.

**Beispiel durchgerechnet.** Die ersten drei Häuser der **Test-Daten**:

| Haus | echter Preis | Vorhersage | Fehler |
| :--- | ---: | ---: | ---: |
| 1 | 132.500 | 224.041 | -91.541 |
| 2 | 415.000 | 473.692 | -58.692 |
| 3 | 494.000 | 421.091 | 72.909 |

- **MAE** dieser drei Häuser: (91.541 + 58.692 + 72.909) / 3 = 74.381 Dollar.
- **RMSE** dieser drei Häuser: Wurzel aus dem Mittel der Quadrate = 75.587 Dollar.
- Der **RMSE** ist etwas größer, weil der größte der drei Fehler im Quadrat mehr Gewicht bekommt.

Auch das **R²** kannst du aus den Tabellen-Werten nachrechnen. Das **Mittelwert-Basis-Modell** hat auf den **Test-Daten** einen **RMSE** von 360.907 Dollar. Da es immer den mittleren Preis der Trainings-Daten vorhersagt, gilt ungefähr $R^2 \approx 1 - (\text{RMSE}_\text{Modell} / \text{RMSE}_\text{Mittelwert})^2 = 1 - (123.490 / 360.907)^2 \approx 0,883$.

In Worten: Das beste Modell hat nur etwa 12 % der quadrierten Fehler des Mittelwerts, es erklärt also etwa 88 % der Preis-Unterschiede.

<a id="s2"></a>

## 2. Zwei Tabellen sicher verknüpfen ([Kapitel 1](zusammenfassung.ipynb#k1))

**Die Idee.** Preis und Haus-Merkmale liegen in zwei Tabellen. Die Spalte `house_id` in `sales` und die Spalte `id` in `details` sind der **Schlüssel**, der sie verbindet. Eine **Verknüpfung** mit `merge` sucht für jeden Verkauf die Zeile des passenden Hauses.

**Warum das funktioniert und wo die Falle liegt.** pandas verbindet jede Zeile links mit jeder passenden Zeile rechts. Kommt ein **Schlüssel** in `details` zweimal vor, entstehen für jeden Verkauf dieses Hauses zwei Zeilen. Das passiert still, ohne Fehler und ohne Warnung. Deshalb prüfst du vorher mit `.is_unique`, ob der **Schlüssel** **eindeutig** ist, und nachher mit einem **Assert**, ob die Zeilen-Zahl stimmt.

Die **Verknüpfungs-Art** im Argument `how` entscheidet nur dann etwas, wenn ein **Schlüssel** in einer Tabelle fehlt. `"left"` behält jeden Verkauf, `"inner"` nur Verkäufe mit Haus, `"right"` jedes Haus und `"outer"` alles.

**Mit den echten Zahlen.**

- `sales` hat 21.597 Zeilen, aber nur 21.420 verschiedene `house_id`.
- `details` hat 21.420 Zeilen und 21.420 verschiedene `id`, der **Schlüssel** ist also **eindeutig**.
- 176 Häuser wurden mehr als einmal verkauft, das ergibt 177 zusätzliche Verkäufe. Die Beziehung ist **Eins-zu-viele**.
- Es gibt 0 Verkäufe ohne Haus und 0 Häuser ohne Verkauf. Jede **Verknüpfungs-Art** liefert hier dasselbe Ergebnis.
- Die neue Tabelle hat 21.597 Zeilen und 23 Spalten: 4 aus `sales` und 19 aus `details`. Die zweite `id` heißt dank `suffixes` jetzt `id_details`.

**Beispiel durchgerechnet.** Das Haus mit `house_id` `795000620` wurde dreimal verkauft:

| Datum | Preis in Dollar |
| :--- | ---: |
| 2014-09-24 | 115.000 |
| 2014-12-15 | 124.000 |
| 2015-03-11 | 157.000 |

Nach der **Verknüpfung** stehen drei Zeilen in `df`, jede mit denselben Haus-Merkmalen und ihrem eigenen Preis. Hätte `details` dieses Haus zweimal enthalten, wären es sechs Zeilen geworden, und das **Assert** hätte gestoppt.

Zum Schluss speicherst du `df` mit `index=False` als `../data/eda.csv`. Ohne `index=False` schreibt pandas die Zeilen-Nummern als zusätzliche Spalte, und beim Einlesen hättest du eine Spalte zu viel.

Worauf achten: Links die zwei Tabellen und ihr **Schlüssel**, rechts wie viele Häuser ein, zwei oder drei Verkäufe haben.

![01_verknuepfung_eins_zu_viele](bilder/01_verknuepfung_eins_zu_viele.png)

<a id="s3"></a>

## 3. Trainings-Daten und Test-Daten ([Kapitel 2.1](zusammenfassung.ipynb#k2-1))

**Die Idee.** Du teilst die Daten einmal mit `train_test_split` in **Trainings-Daten** und **Test-Daten**. Alles Untersuchen, Bauen und Abstimmen passiert auf den **Trainings-Daten**. Die **Test-Daten** öffnest du erst ganz am Ende.

**Warum das funktioniert.** Ein Modell soll Preise von Häusern vorhersagen, die es noch nie gesehen hat. Die **Test-Daten** spielen genau diese neuen Häuser. Sobald du aber eine Entscheidung nach den **Test-Daten** triffst, zum Beispiel welches Modell du nimmst, sind sie nicht mehr neu. Ihr Ergebnis wäre dann zu optimistisch. Deshalb teilst du sogar vor dem Untersuchen: Auch das, was du beim Anschauen lernst, soll nur aus den **Trainings-Daten** kommen.

Ein fester `random_state` sorgt dafür, dass du bei jedem Lauf dieselbe Aufteilung bekommst. Ohne ihn würden sich alle Zahlen bei jedem Neustart ändern. Eigene Validierungs-Daten brauchst du nicht, denn die **Kreuz-Validierung** in Abschnitt 9 übernimmt diese Rolle innerhalb der **Trainings-Daten**.

**Mit den echten Zahlen.** Mit `test_size=0.2` und `random_state=42` entstehen 17.277 Zeilen **Trainings-Daten** und 4.320 Zeilen **Test-Daten**. Beide DataFrames enthalten noch die Spalte `price`, damit **Merkmale** und **Ziel-Variable** zusammen-bleiben.

**Beispiel durchgerechnet.** 20 % von 21.597 sind 4.319,4. scikit-learn rundet auf 4.320 Zeilen auf, es bleiben 21.597 − 4.320 = 17.277 Zeilen zum Trainieren.

Worauf achten: Der orange Teil bleibt bis Schritt 8 gesperrt.

![02_aufteilung_trainings_test](bilder/02_aufteilung_trainings_test.png)

<a id="s4"></a>

## 4. Die Daten untersuchen ([Kapitel 2.1](zusammenfassung.ipynb#k2-1))

**Die Idee.** Bevor du ein Modell baust, beantwortest du auf den **Trainings-Daten** vier Fragen: Welche Spalten hängen mit dem Preis zusammen? Welche sind **schief**? Wo gibt es **fehlende Werte**? Welche **Merkmale** enthalten dieselbe Information?

**Warum das funktioniert.** Jede Antwort führt zu einer Entscheidung. Die stärkste **Korrelation** liefert die Regel für das **Regel-Basis-Modell**. **Fehlende Werte** brauchen ein **Auffüllen**. Doppelte Information kannst du weglassen, denn sie macht lineare Modelle unruhig, ohne Neues zu bringen.

Die **Korrelation** misst nur gerade Zusammenhänge. Ein Wert nahe 0 heißt also nicht, dass eine Spalte nutzlos ist. `zipcode` hat zum Beispiel eine **Korrelation** von -0,05, weil die Post-Leitzahl eine Kategorie ohne sinnvolle Reihenfolge ist. Als Kategorie behandelt, trägt sie trotzdem viel über die Lage bei.

**Mit den echten Zahlen.**

| Frage | Ergebnis |
| :--- | :--- |
| Stärkste **Korrelation** mit `price` | `sqft_living` 0,70, `grade` 0,67, `sqft_above` 0,60, `sqft_living15` 0,59, `bathrooms` 0,53 |
| Am stärksten **schief** | `sqft_lot` 13,37, `waterfront` 11,28, `sqft_lot15` 8,90, `yr_renovated` 4,58, `price` 3,95 |
| **Fehlende Werte** | `yr_renovated` 3.069 Zeilen oder 17,8 %, `waterfront` 1.907 oder 11,0 %, `sqft_basement` 356 oder 2,1 %, `view` 53 oder 0,3 % |
| Gleiche Information | `sqft_living` ist genau `sqft_above` plus `sqft_basement`, `id_details` ist eine Kopie von `house_id` |

Weitere Beobachtungen: `sqft_living` und `sqft_above` haben untereinander eine **Korrelation** von 0,88, `sqft_living` und `grade` eine von 0,76. In `yr_renovated` stehen bei 78,8 % der Häuser eine 0, das heißt "nie renoviert".

**Beispiel durchgerechnet.** Der Preis ist **schief** mit einer Schiefe von 3,95. Das zeigt sich an Mittelwert und Median: Der mittlere Preis der **Trainings-Daten** ist 541.519 Dollar, der Median nur 450.000 Dollar. Wenige sehr teure Häuser, bis zu 7.700.000 Dollar, ziehen den Mittelwert um 91.519 Dollar nach oben.

Worauf achten: Der lange Ausläufer nach rechts und der Abstand zwischen Median und Mittelwert.

![03_preis_verteilung](bilder/03_preis_verteilung.png)

Worauf achten: Der gelbe Balken ist die Spalte für das **Regel-Basis-Modell**, ganz unten steht `zipcode` mit fast 0.

![04_korrelation_mit_preis](bilder/04_korrelation_mit_preis.png)

Worauf achten: `yr_renovated` und `waterfront` haben die meisten Lücken, `view` fast keine.

![05_fehlende_werte](bilder/05_fehlende_werte.png)

<a id="s5"></a>

## 5. Änderungen von Hand und neue Merkmale ([Kapitel 2.1](zusammenfassung.ipynb#k2-1))

**Die Idee.** Manche Änderungen machst du selbst, außerhalb der **Pipeline**: neue Spalten bauen und unnütze Spalten löschen. Das Projekt verlangt, dass du jede dieser Änderungen festhältst. In Schritt 8 machst du genau dieselben Änderungen an den **Test-Daten**.

**Warum das funktioniert.** Das Modell erwartet bei der Vorhersage dieselben Spalten wie beim Training. Fehlt eine Spalte oder ist sie anders berechnet, scheitert die Vorhersage oder wird falsch. Die sicherste Lösung ist eine Funktion `prepare`, die du auf beide Teile anwendest. Dann kann keine Änderung vergessen werden.

Wichtig ist der Unterschied zur **Pipeline**: Die Änderungen in `prepare` lernen nichts aus den Daten. Sie rechnen jede Zeile für sich um. Alles, was aus den Daten gelernt wird, wie ein Median oder ein Mittelwert, gehört dagegen in die **Pipeline**.

**Mit den echten Zahlen.**

| Änderung | Begründung aus Abschnitt 4 |
| :--- | :--- |
| `age_at_sale` = Verkaufs-Jahr minus `yr_built` | das Alter beim Verkauf beschreibt das Haus direkter als das Baujahr |
| `renovated` = 1, wenn `yr_renovated` größer als 0 ist | `yr_renovated` hat 17,8 % Lücken und 78,8 % Nullen |
| `id`, `house_id`, `id_details` löschen | Kennungen sagen nichts über den Preis |
| `date` löschen | als Text unbrauchbar, das Jahr steckt schon in `age_at_sale` |
| `sqft_basement` löschen | gleich `sqft_living` minus `sqft_above`, dazu 2,1 % Lücken |

Danach bleiben 18 **Merkmale**. `age_at_sale` liegt zwischen -1 und 115 Jahren. `renovated` ist bei 3,4 % der **Trainings-Daten** 1. Fehlt `yr_renovated`, wird `renovated` zu 0, denn ein Vergleich mit einem **fehlenden Wert** ist nie wahr.

**Beispiel durchgerechnet.** Ein Haus mit Baujahr 1955, verkauft im Oktober 2014, bekommt `age_at_sale` = 2014 − 1955 = 59. Der kleinste Wert -1 entsteht, wenn ein Haus ein Jahr vor seinem eingetragenen Baujahr verkauft wurde, also vermutlich als Neubau.

<a id="s6"></a>

## 6. Basis-Modelle als Messlatte ([Kapitel 2.1](zusammenfassung.ipynb#k2-1))

**Die Idee.** Bevor du ein echtes Modell baust, baust du zwei sehr einfache: das **Mittelwert-Basis-Modell** und das **Regel-Basis-Modell**. Jedes echte Modell muss beide schlagen.

**Warum das funktioniert.** Eine Zahl wie "RMSE 160.000 Dollar" sagt allein wenig. Erst der Vergleich zeigt, ob das Modell etwas gelernt hat. Das **Mittelwert-Basis-Modell** ist die unterste Grenze: Es kennt kein einziges **Merkmal**. Das **Regel-Basis-Modell** ist die Grenze für "gesunden Menschen-Verstand": eine Regel, die jeder Makler im Kopf rechnen kann. Die Werte beider Regeln lernst du nur auf den **Trainings-Daten**.

**Mit den echten Zahlen.** Der Median-Preis pro Quadrat-Fuß in den **Trainings-Daten** ist 244,44 Dollar. Daraus entsteht die Regel:

$$\hat{y} = 244,44 \cdot \text{sqft\_living}$$

In Worten: Der vorhergesagte Preis ist 244,44 Dollar mal die Wohn-Fläche in Quadrat-Fuß.

| **Basis-Modell** | **RMSE** Training | **RMSE** Test | **R²** Test |
| :--- | ---: | ---: | ---: |
| **Mittelwert-Basis-Modell** | 368.958 | 360.907 | 0,00 |
| **Regel-Basis-Modell** | 267.068 | 260.103 | 0,48 |

Die einfache Regel erklärt schon 48 % der Preis-Unterschiede. Ein echtes Modell muss also deutlich unter 260.103 Dollar **RMSE** kommen.

**Beispiel durchgerechnet.** Das erste Test-Haus hat 850 Quadrat-Fuß. Die Regel sagt 244,44 · 850 = 207.778 Dollar vorher. Das **Mittelwert-Basis-Modell** sagt 541.519 Dollar vorher. Der echte Preis war 132.500 Dollar, die Regel liegt also viel näher.

Worauf achten: Die schräge Linie folgt der Punkte-Wolke, die waagrechte Linie ignoriert die Fläche.

![06_basis_modelle](bilder/06_basis_modelle.png)

<a id="s7"></a>

## 7. Pipeline und ColumnTransformer ([Kapitel 2.2](zusammenfassung.ipynb#k2-2))

**Die Idee.** Eine **Pipeline** verkettet alle gelernten Vorbereitungs-Schritte und das Modell zu einem Objekt. Ein **ColumnTransformer** schickt dabei verschiedene Spalten durch verschiedene Schritte: Zahlen-Spalten bekommen **Auffüllen** mit dem Median und **Skalieren**, die Kategorie `zipcode` bekommt **Auffüllen** mit dem häufigsten Wert und **One-Hot-Kodierung**.

**Warum das funktioniert.** Jeder dieser Schritte lernt etwas aus den Daten: den Median, den Mittelwert, die Standard-Abweichung, die Liste der Post-Leitzahlen. Würdest du das vor der Aufteilung auf allen Daten lernen, steckte Wissen aus den **Test-Daten** im Modell. Die **Pipeline** verhindert das: `fit` lernt nur auf den Daten, die du übergibst, und `predict` wendet die gelernten Werte nur an. In der **Kreuz-Validierung** lernt die **Pipeline** sogar in jedem **Fold** neu, nur aus dessen Trainings-Teil.

**Skalieren** ist für **Ridge** wichtig, weil die **Regularisierung** alle Koeffizienten gleich stark bestraft. Ohne **Skalieren** hätte eine Fläche in Quadrat-Fuß einen winzigen Koeffizienten und die Zahl der Stockwerke einen großen, und die Strafe träfe sie ungleich.

`zipcode` bekommt **One-Hot-Kodierung**, weil die Post-Leitzahl eine Kategorie ist. Eine größere Post-Leitzahl bedeutet kein größeres oder teureres Haus. Mit `handle_unknown="ignore"` bekommt eine Post-Leitzahl, die im Training fehlte, einfach überall 0 statt eines Fehlers.

**Mit den echten Zahlen.** Aus 18 **Merkmalen** werden 17 Zahlen-Spalten und eine Kategorie-Spalte. `zipcode` hat 70 Werte, die **One-Hot-Kodierung** macht daraus 70 Spalten. Zusammen sieht **Ridge** 17 + 70 = 87 Spalten.

**Beispiel durchgerechnet.** In den **Trainings-Daten** hat `sqft_living` den Mittelwert 2.083,5 und die Standard-Abweichung 917,5. Ein Haus mit 3.000 Quadrat-Fuß wird beim **Skalieren** zu (3.000 − 2.083,5) / 917,5 = 1,00. In Worten: Es liegt etwa eine Standard-Abweichung über dem Durchschnitt.

Worauf achten: Die zwei Wege im gestrichelten Kasten und die Spalten-Zahl vor **Ridge**.

![07_pipeline_aufbau](bilder/07_pipeline_aufbau.png)

<a id="s8"></a>

## 8. Ridge, Regularisierung und Polynom-Merkmale ([Kapitel 2.2](zusammenfassung.ipynb#k2-2))

**Die Idee.** **Ridge** ist eine lineare **Regression** mit **Regularisierung**. Sie sucht Koeffizienten, die die quadrierten Fehler klein machen, und bestraft dabei zusätzlich große Koeffizienten. Der **Hyper-Parameter** `alpha` legt fest, wie stark die Strafe ist. **Polynom-Merkmale** ergänzen alle Quadrate und Produkte der Zahlen-Spalten, damit ein lineares Modell auch Kurven und Wechsel-Wirkungen abbilden kann.

$$\min_w \; \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j w_j^2$$

In Worten: **Ridge** sucht die Koeffizienten $w$, bei denen Fehler-Summe plus `alpha` mal Summe der quadrierten Koeffizienten am kleinsten ist.

**Warum das funktioniert.** Viele **Merkmale** hängen eng zusammen, zum Beispiel `sqft_living` und `sqft_above` mit 0,88. Eine lineare **Regression** ohne Strafe kann dann große Koeffizienten mit entgegen-gesetzten Vorzeichen wählen, die sich fast aufheben. Die Strafe hält die Koeffizienten klein und das Modell ruhig. Bei vielen Spalten, wie mit **Polynom-Merkmalen**, schützt sie außerdem vor **Über-Anpassung**.

**Polynom-Merkmale** helfen, weil der Preis nicht überall gleich mit der Fläche wächst. Das Produkt aus Fläche und Note kann zum Beispiel ausdrücken, dass ein Quadrat-Fuß in einem gut bewerteten Haus mehr wert ist.

**Mit den echten Zahlen.**

| Modell | Spalten | Bestes `alpha` | **RMSE** der **Kreuz-Validierung** |
| :--- | ---: | ---: | ---: |
| **Ridge** | 87 | jeder Wert bis 0,3 | ≈ 161.500 |
| **Ridge** mit **Polynom-Merkmalen** | 240 | 0,00055 | 132.572 |

Bei **Ridge** ohne **Polynom-Merkmale** ändert `alpha` bis 0,3 den Fehler um weniger als 10 Dollar. Mit 17.277 Zeilen und nur 87 Spalten braucht das Modell kaum Strafe. Deshalb kann das "beste" `alpha` von Lauf zu Lauf anders aussehen, und die **Ridge**-Werte sind hier auf Hunderter gerundet. Ab `alpha` = 1 wird der Fehler größer, bei `alpha` = 100 liegt er schon bei 172.850 Dollar.

**Beispiel durchgerechnet.** Wie entstehen 240 Spalten? `PolynomialFeatures(degree=2, include_bias=False)` macht aus 17 Zahlen-Spalten:

- die 17 Original-Spalten,
- 17 Quadrate,
- 136 Produkte aus je zwei verschiedenen Spalten.

Das sind 17 + 17 + 136 = 170 Spalten. Dazu kommen die 70 Spalten aus `zipcode`, zusammen 240.

<a id="s9"></a>

## 9. Kreuz-Validierung, Zufalls-Suche und Gitter-Suche ([Kapitel 2.2](zusammenfassung.ipynb#k2-2))

**Die Idee.** Um `alpha` zu wählen, brauchst du eine Bewertung, die nicht die **Test-Daten** benutzt. Die **Kreuz-Validierung** teilt die **Trainings-Daten** in fünf **Folds**. Jeder **Fold** ist einmal der Validierungs-Teil, die anderen vier sind der Trainings-Teil. Der Mittelwert der fünf Fehler ist der Wert der **Kreuz-Validierung**. Die **Zufalls-Suche** probiert damit viele zufällige `alpha` aus einem breiten Bereich aus. Die **Gitter-Suche** probiert danach jeden Wert aus einer engen Liste um den besten Fund.

**Warum das funktioniert.** Ein einzelner Validierungs-Teil kann Glück oder Pech haben. Fünf Teile mitteln diesen Zufall heraus, und jede Zeile wird genau einmal zum Bewerten genutzt. Die zweistufige Suche spart Rechenzeit: `loguniform(1e-4, 1e4)` verteilt die Versuche gleichmäßig über acht Zehner-Potenzen, und erst wenn klar ist, in welcher Gegend das Optimum liegt, sucht die **Gitter-Suche** fein. Liegt das beste `alpha` am Rand des Gitters, war das Gitter zu eng.

scikit-learn macht einen Wert immer so groß wie möglich. Deshalb heißt die **Metrik** `neg_root_mean_squared_error` und ist negativ. Ein Minus vor `best_score_` ergibt wieder den **RMSE**.

**Mit den echten Zahlen.** `KFold(n_splits=5, shuffle=True, random_state=42)` teilt die 17.277 Zeilen in **Folds** mit 3.456, 3.456, 3.455, 3.455 und 3.455 Zeilen. Für **Ridge** mit **Polynom-Merkmalen** fand die **Zufalls-Suche** mit 30 Versuchen `alpha` = 0,00029 mit 132.587 Dollar. Die **Gitter-Suche** verbesserte das auf `alpha` = 0,00055 mit 132.572 Dollar, im Gitter innen und nicht am Rand.

**Beispiel durchgerechnet.** Im ersten Durchgang trainiert das Modell auf 13.821 Zeilen und wird auf 3.456 Zeilen bewertet. Eine Suche mit 30 `alpha`-Werten und 5 **Folds** trainiert also 30 · 5 = 150 Modelle, die **Gitter-Suche** mit 11 Werten noch einmal 55.

Worauf achten: Kein Durchgang nutzt die **Test-Daten**, und jeder **Fold** ist genau einmal hellblau.

![08_kreuz_validierung](bilder/08_kreuz_validierung.png)

Worauf achten: Die flache Kurve von **Ridge** und das klare Minimum mit **Polynom-Merkmalen** links unten.

![09_alpha_suche](bilder/09_alpha_suche.png)

<a id="s10"></a>

## 10. Bewerten auf den Test-Daten und Residuen ([Kapitel 2.2](zusammenfassung.ipynb#k2-2))

**Die Idee.** Erst jetzt öffnest du die **Test-Daten**. Du wendest `prepare` an, teilst in `X_test` und `y_test` und berechnest für jedes Modell Trainings-Werte und Test-Werte in einer Tabelle. Danach untersuchst du die **Residuen** des besten Modells.

**Warum das funktioniert.** Der Vergleich in einer Tabelle beantwortet zwei Fragen auf einmal. Erstens: Schlägt das Modell die **Basis-Modelle**? Zweitens: Ist der Test-Fehler viel größer als der Trainings-Fehler? Das wäre **Über-Anpassung**. Die **Residuen** zeigen dann, wo das Modell falsch liegt, denn eine einzelne Zahl wie der **RMSE** verrät nicht, ob alle Häuser ein bisschen oder wenige Häuser sehr falsch geschätzt werden.

$$e_i = y_i - \hat{y}_i$$

In Worten: Ein positives **Residuum** heißt, das Modell hat zu niedrig geschätzt, ein negatives heißt, es hat zu hoch geschätzt.

**Mit den echten Zahlen.** Alle Werte in Dollar, **Ridge** auf Hunderter gerundet:

| Modell | **RMSE** Training | **RMSE** Test | **MAE** Test | **R²** Test |
| :--- | ---: | ---: | ---: | ---: |
| **Mittelwert-Basis-Modell** | 368.958 | 360.907 | 231.542 | 0,00 |
| **Regel-Basis-Modell** | 267.068 | 260.103 | 165.076 | 0,48 |
| **Ridge** | ≈ 160.200 | ≈ 161.700 | ≈ 96.300 | 0,80 |
| **Ridge** mit **Polynom-Merkmalen** | 125.303 | 123.490 | 75.765 | 0,88 |

- Beide **Ridge**-Modelle schlagen beide **Basis-Modelle** deutlich.
- Beim besten Modell ist der Test-Fehler sogar 1.814 Dollar kleiner als der Trainings-Fehler. Es gibt kein Zeichen von **Über-Anpassung**.
- Das mittlere **Residuum** ist -1.116 Dollar, der Median 108 Dollar. Das Modell schätzt also im Mittel weder zu hoch noch zu niedrig.
- 298 Test-Häuser oder 6,9 % liegen mehr als 200.000 Dollar daneben.

| Preis-Viertel der **Test-Daten** | mittleres absolutes **Residuum** in Dollar |
| :--- | ---: |
| bis 319.800 | 44.972 |
| bis 450.000 | 52.729 |
| bis 637.500 | 68.239 |
| bis 7.060.000 | 137.264 |

**Beispiel durchgerechnet.** Der größte Fehler: ein Haus mit 7.120 Quadrat-Fuß, Note 12, Post-Leitzahl `98006`. Es wurde für 900.000 Dollar verkauft, das Modell sagte 2.726.355 Dollar vorher. Das **Residuum** ist 900.000 − 2.726.355 = -1.826.355 Dollar. Sehr große, gut bewertete Häuser sind selten, deshalb hat das Modell für sie wenig Beispiele, und Quadrate und Produkte sehr großer Werte können die Vorhersage weit nach oben oder unten ziehen.

Im teuersten Viertel ist der mittlere Fehler mit 137.264 Dollar etwa 3-mal so groß wie im günstigsten mit 44.972 Dollar. Für teure Häuser ist das Modell in Dollar gemessen am unsichersten.

Worauf achten: Die Paare aus blauem und orangem Balken sind fast gleich hoch.

![10_modell_vergleich](bilder/10_modell_vergleich.png)

Worauf achten: Links die schmale Spitze um 0, rechts der hohe Balken im teuersten Viertel.

![11_residuen](bilder/11_residuen.png)

<a id="denkfehler"></a>

## Typische Denkfehler

| Denkfehler | Richtig ist |
| :--- | :--- |
| "pandas warnt mich, wenn eine Verknüpfung Zeilen verdoppelt." | pandas kopiert Zeilen still. Prüfe den **Schlüssel** mit `.is_unique` und die Zeilen-Zahl mit einem **Assert**. |
| "Ich schaue mir erst alle Daten an und teile dann." | Teile zuerst. Auch Erkenntnisse aus dem Anschauen sollen nur aus den **Trainings-Daten** kommen. |
| "Ich wähle das Modell mit dem besten Test-Wert." | Wähle mit der **Kreuz-Validierung**. Die **Test-Daten** sind nur für die letzte Prüfung da. |
| "Den Median zum Auffüllen rechne ich einmal auf allen Daten aus." | Gelernte Werte gehören in die **Pipeline**, damit sie nur aus den **Trainings-Daten** kommen. |
| "`zipcode` ist eine Zahl, also skaliere ich sie." | Die Post-Leitzahl ist eine Kategorie und bekommt **One-Hot-Kodierung**. |
| "Eine Korrelation nahe 0 heißt: Die Spalte ist nutzlos." | Die **Korrelation** misst nur gerade Zusammenhänge. `zipcode` hilft als Kategorie trotzdem viel. |
| "Mehr Spalten machen ein Modell automatisch besser." | Mehr Spalten können zu **Über-Anpassung** führen. Erst der Vergleich von Trainings-Werten und Test-Werten zeigt es. |
| "Ein negativer `best_score_` ist ein Fehler im Code." | scikit-learn maximiert Werte, deshalb ist der Fehler negativ. Ein Minus davor ergibt den **RMSE**. |
| "Ein guter RMSE heißt, das Modell ist überall gleich gut." | Die **Residuen** zeigen: Bei teuren Häusern ist der Fehler etwa dreimal so groß. |
| "Die Änderungen von Hand an den Test-Daten mache ich später nochmal neu." | Schreib sie in eine Funktion und wende sie auf beide Teile an, sonst passen die Spalten nicht. |

<a id="spick"></a>

## Code-Spickzettel nach Thema

**Laden und verknüpfen**

```python
sales = pd.read_csv("../data/king_county_house_sales.csv")
details = pd.read_csv("../data/king_county_house_details.csv")
sales["house_id"].nunique()                      # verschiedene Häuser in sales
details["id"].is_unique                          # True: Schlüssel ist eindeutig
(~sales["house_id"].isin(details["id"])).sum()   # Verkäufe ohne Haus
df = sales.merge(details, left_on="house_id", right_on="id", how="left", suffixes=("", "_details"))
assert len(df) == len(sales)
df.to_csv("../data/eda.csv", index=False)
```

**Aufteilen und untersuchen**

```python
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train.corr(numeric_only=True)["price"].sort_values(ascending=False)
df_train.skew(numeric_only=True)
df_train.isna().sum()
```

**Änderungen von Hand und Basis-Modelle**

```python
def prepare(data):
    data = data.copy()
    data["age_at_sale"] = pd.to_datetime(data["date"]).dt.year - data["yr_built"]
    data["renovated"] = (data["yr_renovated"] > 0).astype(int)
    return data.drop(columns=["id", "house_id", "id_details", "date", "yr_renovated", "sqft_basement"])

mean_baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
price_per_sqft = (y_train / X_train["sqft_living"]).median()   # Regel lernen
```

**Pipeline**

```python
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), ["zipcode"]),
])
ridge = Pipeline([("preprocess", preprocess), ("model", Ridge())])
# Polynom-Merkmale: PolynomialFeatures(degree=2, include_bias=False) vor StandardScaler einfügen
```

**Abstimmen**

```python
cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(ridge, {"model__alpha": loguniform(1e-4, 1e4)}, n_iter=30, cv=cv,
                            scoring="neg_root_mean_squared_error", random_state=42).fit(X_train, y_train)
best = search.best_params_["model__alpha"]
grid = GridSearchCV(ridge, {"model__alpha": np.geomspace(best / 5, best * 5, 11)}, cv=cv,
                    scoring="neg_root_mean_squared_error").fit(X_train, y_train)
-grid.best_score_                                 # RMSE der Kreuz-Validierung
```

**Bewerten**

```python
pred = grid.best_estimator_.predict(X_test)
root_mean_squared_error(y_test, pred), mean_absolute_error(y_test, pred), r2_score(y_test, pred)
residuals = y_test - pred
residuals.abs().groupby(pd.qcut(y_test, 4)).mean()   # Fehler pro Preis-Viertel
```

<a id="pruefen"></a>

## Prüfe dich selbst

**Frage 1 von 13:** Warum hat die verknüpfte Tabelle genauso viele Zeilen wie `sales`?

<details>
<summary>Antwort</summary>

Weil der **Schlüssel** `id` in `details` **eindeutig** ist. Jeder Verkauf findet höchstens ein Haus, und weil kein Haus fehlt, genau eines. So entstehen 21.597 Zeilen.

</details>

**Frage 2 von 13:** Was würde passieren, wenn ein Haus zweimal in `details` stünde?

<details>
<summary>Antwort</summary>

Jeder Verkauf dieses Hauses würde verdoppelt. pandas meldet das nicht, erst das **Assert** würde stoppen.

</details>

**Frage 3 von 13:** Warum teilst du die Daten, bevor du sie untersuchst?

<details>
<summary>Antwort</summary>

Damit alles, was du aus den Daten lernst, nur aus den **Trainings-Daten** kommt. Sonst wären die **Test-Daten** am Ende nicht mehr neu.

</details>

**Frage 4 von 13:** Warum ist der **RMSE** des besten Modells größer als sein **MAE**?

<details>
<summary>Antwort</summary>

Der **RMSE** quadriert die Fehler. Die wenigen sehr großen Fehler zählen dadurch stärker, deshalb liegt er mit 123.490 Dollar über dem **MAE** mit 75.765 Dollar.

</details>

**Frage 5 von 13:** Was sagt ein **R²** von etwa 0 beim **Mittelwert-Basis-Modell**?

<details>
<summary>Antwort</summary>

Es erklärt keinen Preis-Unterschied. Das ist zu erwarten, denn das **R²** vergleicht genau mit dieser Vorhersage.

</details>

**Frage 6 von 13:** Warum löschst du `sqft_basement`?

<details>
<summary>Antwort</summary>

Weil sie genau `sqft_living` minus `sqft_above` ist und 2,1 % **fehlende Werte** hat.

</details>

**Frage 7 von 13:** Was ist der Unterschied zwischen `prepare` und der **Pipeline**?

<details>
<summary>Antwort</summary>

`prepare` rechnet jede Zeile für sich um und lernt nichts. Die **Pipeline** lernt Werte wie Median und Mittelwert, und zwar nur aus den Daten, die du an `fit` gibst.

</details>

**Frage 8 von 13:** Warum braucht **Ridge** **Skalieren**?

<details>
<summary>Antwort</summary>

Die **Regularisierung** bestraft alle Koeffizienten gleich. Nur wenn alle Spalten dieselbe Skala haben, trifft die Strafe sie fair.

</details>

**Frage 9 von 13:** Wie kommen 240 Spalten mit **Polynom-Merkmalen** zustande?

<details>
<summary>Antwort</summary>

17 Original-Spalten, 17 Quadrate und 136 Produkte ergeben 170, dazu 70 Spalten aus `zipcode`.

</details>

**Frage 10 von 13:** Warum ist `best_score_` negativ?

<details>
<summary>Antwort</summary>

scikit-learn macht einen Wert immer so groß wie möglich. Deshalb gibt `neg_root_mean_squared_error` den Fehler mit Minus zurück.

</details>

**Frage 11 von 13:** Warum kann das beste `alpha` für **Ridge** von Lauf zu Lauf anders sein?

<details>
<summary>Antwort</summary>

Für jedes `alpha` bis 0,3 ist der Fehler fast gleich, die Unterschiede sind nur wenige Dollar. Winzige Rechen-Unterschiede entscheiden dann, welcher Wert gewinnt.

</details>

**Frage 12 von 13:** Woran erkennst du, dass das beste Modell nicht überangepasst ist?

<details>
<summary>Antwort</summary>

Trainings-Fehler 125.303 Dollar und Test-Fehler 123.490 Dollar liegen nah beieinander.

</details>

**Frage 13 von 13:** Bei welchen Häusern liegt das beste Modell am meisten daneben?

<details>
<summary>Antwort</summary>

Bei teuren Häusern. Im teuersten Viertel ist der mittlere Fehler 137.264 Dollar, im günstigsten 44.972 Dollar.

</details>

<a id="faden"></a>

## Der rote Faden in einem Satz

Du verknüpfst zwei Tabellen sicher über einen **eindeutigen** **Schlüssel**, sperrst sofort die **Test-Daten** weg, baust auf den **Trainings-Daten** zwei **Basis-Modelle** und eine **Pipeline** mit **Ridge**, wählst `alpha` mit **Kreuz-Validierung** und prüfst erst ganz am Ende auf den **Test-Daten**, dass **Ridge** mit **Polynom-Merkmalen** mit 123.490 Dollar **RMSE** beide **Basis-Modelle** schlägt.

Worauf achten: Das ist derselbe Weg wie im Bild, vom ersten zum letzten Kasten.

![12_roter_faden](bilder/12_roter_faden.png)

[⬆ Zum Inhalt](#inhalt)